# SpecCoder Educational Notebook 01 — Understanding the SpecParser
*Learn how the SpecParser module transforms OpenSpec YAML into structured Python dataclasses.*

**What you'll learn**
- How the SpecParser loads YAML and produces typed Python dataclasses (OpenSpec, Requirement, etc.).
- The purpose of key methods: `parse_file`, `parse_string`, `_parse_data`, `_parse_requirement`, `validate_spec`.
- How to run the parser on a real spec (we'll use `rhythmic_pulse_generator.yml`) and inspect results.
- How to format requirements and function signatures for downstream prompts.

**Educational approach – DONE MEANS TAUGHT**
We alternate: *Explain → Show real source → Run a concrete example*.

> Note: In your repo, prefer the full package import path per the Notebook Development Guide.
> Here we include a small fallback so this tutorial can execute standalone if the package path isn't available.

In [1]:
from pathlib import Path
import sys
import inspect
from textwrap import indent

# --- Repo-root resolution (preferred pattern in your repo) ---
current_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
repo_root = current_dir.parent.parent.parent.parent  # adjust if storing under notebooks/
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# --- Preferred import: package path ---
# In your repo, this should succeed:
try:
    from asabaal_utils.agents.spec_coder.spec_parser import SpecParser
    USING_PACKAGE_IMPORT = True
except Exception:
    # Fallback for standalone execution in this environment only
    USING_PACKAGE_IMPORT = False
    if str(current_dir) not in sys.path:
        sys.path.insert(0, str(current_dir))
    # Local module fallback
    from spec_parser import SpecParser  # noqa: E402

print("Using package import:", USING_PACKAGE_IMPORT)

Using package import: True


## Phase 1. Conceptual overview
The SpecParser converts **OpenSpec-style YAML** into structured Python objects:

- **OpenSpec**: overall metadata and a list of **Requirement** items.
- **Requirement**: id, title, description, optional interface and validation.
- **ValidationCriteria**: type, file, target of tests.
- **FunctionInterface**: function name, parameters, returns, and optional example.

We will parse a real file: `rhythmic_pulse_generator.yml`.
This spec lists high-level requirements (generate pulse, apply envelope, export audio)
and a separate `interfaces` block that describes a class API.

## Phase 2. Code walkthrough (selected methods)
We'll peek at key methods directly from the loaded class and annotate what they do.

In [2]:
def show_source(obj, header: str):
    print(f"### {header}\n")
    try:
        src = inspect.getsource(obj)
        print(src)
    except OSError:
        print("Source unavailable (possibly C-accelerated or dynamically defined).")

show_source(SpecParser.__init__, "SpecParser.__init__")

### SpecParser.__init__

    def __init__(self):
        self.specs: Dict[str, OpenSpec] = {}



In [3]:
show_source(SpecParser.parse_file, "SpecParser.parse_file")

### SpecParser.parse_file

    def parse_file(self, file_path: Path) -> OpenSpec:
        """Parse a single OpenSpec YAML file."""
        with open(file_path, 'r', encoding='utf-8') as f:
            data = yaml.safe_load(f)
        
        return self._parse_data(data, str(file_path))



In [4]:
show_source(SpecParser.parse_string, "SpecParser.parse_string")

### SpecParser.parse_string

    def parse_string(self, yaml_content: str, source: str = "<string>") -> OpenSpec:
        """Parse OpenSpec YAML from a string."""
        data = yaml.safe_load(yaml_content)
        return self._parse_data(data, source)



In [5]:
show_source(SpecParser._parse_data, "SpecParser._parse_data")

### SpecParser._parse_data

    def _parse_data(self, data: Dict[str, Any], source: str) -> OpenSpec:
        """Parse raw YAML data into OpenSpec object."""
        # Extract basic metadata
        spec_id = data.get('name', data.get('spec_id', 'unknown'))
        title = data.get('title', data.get('description', 'Untitled Specification'))
        version = data.get('version', '0.1.0')
        
        # Parse requirements
        requirements = []
        for req_data in data.get('requirements', []):
            requirement = self._parse_requirement(req_data)
            requirements.append(requirement)
        
        return OpenSpec(
            spec_id=spec_id,
            title=title,
            version=version,
            requirements=requirements,
            raw_data=data
        )



In [6]:
show_source(SpecParser._parse_requirement, "SpecParser._parse_requirement")

### SpecParser._parse_requirement

    def _parse_requirement(self, req_data: Dict[str, Any]) -> Requirement:
        """Parse a single requirement from YAML data."""
        req_id = req_data.get('name', req_data.get('id', 'unknown'))
        title = req_data.get('title', req_data.get('name', 'Untitled Requirement'))
        description = req_data.get('description', '')
        
        # Parse interface if present
        interface = None
        if 'interface' in req_data:
            interface_data = req_data['interface']
            interface = FunctionInterface(
                function=interface_data.get('function', ''),
                parameters=interface_data.get('parameters', {}),
                returns=interface_data.get('returns', ''),
                example=interface_data.get('example')
            )
        
        # Parse validation criteria
        validation = []
        validation_data = req_data.get('validation', [])
        
        # Handle both string and list

In [7]:
show_source(SpecParser.validate_spec, "SpecParser.validate_spec")

### SpecParser.validate_spec

    def validate_spec(self, spec: OpenSpec) -> List[str]:
        """Validate the specification and return any issues."""
        issues = []
        
        if not spec.spec_id:
            issues.append("Missing spec_id")
        
        if not spec.title:
            issues.append("Missing title")
        
        if not spec.requirements:
            issues.append("No requirements found")
        
        for i, req in enumerate(spec.requirements):
            if not req.id:
                issues.append(f"Requirement {i+1} missing ID")
            if not req.title:
                issues.append(f"Requirement {i+1} missing title")
            if not req.description:
                issues.append(f"Requirement {i+1} missing description")
        
        return issues



**Notes:**
- `_parse_data` extracts top-level fields `name/spec_id`, `title/description`, `version`,
  then iterates over `requirements` to build `Requirement` instances.
- `_parse_requirement` pulls id/title/description and optionally parses `interface` and `validation` fields
  **when those are provided inside each requirement**.
- If your spec declares `interfaces` at the top level (separate from each requirement),
  the current parser will not automatically attach those to requirements — it expects per-requirement `interface` data.

We'll see the effect of that in the live demo.

## Phase 3. Practical demonstration with your YAML
Let's parse `/mnt/data/rhythmic_pulse_generator.yml` and explore.

In [8]:
from importlib import resources
from pathlib import Path

parser = SpecParser()

# Determine package name dynamically; fallback to file location if run as script
package_name = __package__ or "asabaal_utils.agents.spec_coder"

try:
    with resources.as_file(
        resources.files(package_name) / "rhythmic_pulse_generator.yml"
    ) as yaml_path:
        spec = parser.parse_file(yaml_path)
except (AttributeError, ModuleNotFoundError):
    # Fallback for direct script execution: look relative to this file
    local_yaml = Path(__file__).parent / "rhythmic_pulse_generator.yml"
    spec = parser.parse_file(local_yaml)

print("Spec ID:", spec.spec_id)
print("Title  :", spec.title)
print("Version:", spec.version)
print("Num requirements:", len(spec.requirements))

Spec ID: rhythmic_pulse_generator
Title  : Rhythmic Pulse Generator
Version: 1.0.0
Num requirements: 3


In [9]:
import textwrap
import yaml

# Display the raw YAML file content (as shipped with the package or locally)
try:
    # Use the same resolved yaml_path from earlier
    with open(yaml_path, "r", encoding="utf-8") as f:
        raw_yaml = f.read()
except NameError:
    # Fallback if yaml_path variable not in scope (e.g., after as_file block)
    from pathlib import Path
    raw_yaml = Path(__file__).with_name("rhythmic_pulse_generator.yml").read_text(encoding="utf-8")

print("=== RAW YAML CONTENT ===\n")
print(textwrap.indent(raw_yaml, "  "))

# Parse YAML again for demonstration
parsed_yaml = yaml.safe_load(raw_yaml)
print("\n=== TOP-LEVEL YAML KEYS ===", list(parsed_yaml.keys()))

# Show parsed first requirement if present
if spec.requirements:
    first = spec.requirements[0]
    print("\n=== FIRST PARSED REQUIREMENT ===")
    print("  id         :", first.id)
    print("  title      :", first.title)
    print("  description:", first.description)
    print("  interface  :", getattr(first, "interface", None))
    print("  validation :", getattr(first, "validation", None))
else:
    print("No requirements found in spec.")


=== RAW YAML CONTENT ===

  spec_id: rhythmic_pulse_generator
  title: Rhythmic Pulse Generator
  version: 1.0.0
  description: Generates rhythmic pulse patterns for audio synthesis
  requirements:
    - id: generate_pulse
      title: Generate Pulse
      description: Generate a rhythmic pulse pattern with specified BPM and duration
      interface:
        function: generate_pulse
        parameters:
          duration: float
        returns: Any
    - id: apply_envelope
      title: Apply Envelope
      description: Apply ADSR envelope to the pulse pattern
      interface:
        function: apply_envelope
        parameters:
          pulse: Any
          attack: float
          decay: float
          sustain: float
          release: float
        returns: Any
    - id: export_audio
      title: Export Audio
      description: Export the generated pattern as audio file
      interface:
        function: export_audio
        parameters:
          pattern: Any
          filename: str

### Format helpers
The parser includes helpers to format content for prompts or scaffolding.

In [10]:
# Show the prompt-formatted summary of requirements (if any)
try:
    print("=== format_requirements_for_prompt ===")
    print(parser.format_requirements_for_prompt(spec))
except Exception as e:
    print("format_requirements_for_prompt error:", e)

=== format_requirements_for_prompt ===
- generate_pulse: Generate Pulse
  Description: Generate a rhythmic pulse pattern with specified BPM and duration

- apply_envelope: Apply Envelope
  Description: Apply ADSR envelope to the pulse pattern

- export_audio: Export Audio
  Description: Export the generated pattern as audio file



In [11]:
# Demonstrate format_function_signature on each requirement.
# Because this YAML does not embed 'interface' blocks per requirement,
# the function will fall back to a placeholder signature.
for req in spec.requirements:
    sig = parser.format_function_signature(req)
    print(f"\nSignature for requirement {req.id}:\n{sig}")


Signature for requirement generate_pulse:
def generate_pulse(duration: float) -> Any:
    """TODO generate_pulse: Generate a rhythmic pulse pattern with specified BPM and duration."""
    pass

Signature for requirement apply_envelope:
def apply_envelope(pulse: Any, attack: float, decay: float, sustain: float, release: float) -> Any:
    """TODO apply_envelope: Apply ADSR envelope to the pulse pattern."""
    pass

Signature for requirement export_audio:
def export_audio(pattern: Any, filename: str) -> None:
    """TODO export_audio: Export the generated pattern as audio file."""
    pass


In [12]:
# Show validate_spec results
issues = parser.validate_spec(spec)
print("\nValidation issues:", issues if issues else "No issues found.")


Validation issues: No issues found.


### Discussion
- Your YAML separates `interfaces` at the top level. The current parser expects any interface data
  to live **inside each requirement**. Therefore, `format_function_signature` will produce a placeholder
  unless `requirement.interface` is present.

**Options:**
1. **Keep parser as-is** and embed an `interface` block inside each requirement that needs one.
2. **Extend SpecParser** to map top-level `interfaces` into the corresponding requirements by name.

We'll include a short exercise below for approach 2.

## Phase 4. Reflection & exercises

**Exercise A (Parser extension):**
- Modify `_parse_data` to read a top-level `interfaces` list.
- Build a mapping by interface `name`.
- When constructing each requirement in `_parse_requirement`, if there is a matching interface by name,
  attach a `FunctionInterface` with parameters/returns/signature parsed from that interface.

**Exercise B (Validation enhancement):**
- Update `validate_spec` to ensure each requirement has either a `description` or an attached `interface`.

**Exercise C (Signature enrichment):**
- Extend `format_function_signature` so that if a requirement has an attached interface,
  you parse the interface's `signature` string into `function` + typed parameters + return type (if present).

**Your turn:** Try Exercise A, then re-run the demo cells to observe improved signatures.